In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

In [3]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

C:\Egyetem\MSC\Survival-Analysis-v2


In [4]:
PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"

DATASETS = {
    "synthetic_binary": DATA_DIR / "synthetic_binary",
    "synthetic_competing": DATA_DIR / "synthetic_competing",
    "SEER": DATA_DIR / "SEER",
    "MNB": DATA_DIR / "MNB",
}

## Tests

In [5]:
from src.datasets.loaders import load_dataset, print_dataset_summary

for name in ["synthetic_binary", "synthetic_competing", "SEER", "MNB"]:
    ds = load_dataset(name, data_root="../data")
    print_dataset_summary(ds)

Dataset: synthetic_binary
Features: 12
Events: [0, 1]
Competing risk: False
n_events: 1

train:
  X: (105000, 12)
  time: (105000,) min: 0.5 max: 890.53345
  event: (105000,) {np.int64(0): np.int64(52500), np.int64(1): np.int64(52500)}

val:
  X: (22500, 12)
  time: (22500,) min: 0.5 max: 447.2063
  event: (22500,) {np.int64(0): np.int64(11250), np.int64(1): np.int64(11250)}

test:
  X: (22500, 12)
  time: (22500,) min: 0.5 max: 416.31116
  event: (22500,) {np.int64(0): np.int64(11250), np.int64(1): np.int64(11250)}
Dataset: synthetic_competing
Features: 12
Events: [0, 1, 2]
Competing risk: True
n_events: 2

train:
  X: (105000, 12)
  time: (105000,) min: 0.5 max: 890.53345
  event: (105000,) {np.int64(0): np.int64(52500), np.int64(1): np.int64(26195), np.int64(2): np.int64(26305)}

val:
  X: (22500, 12)
  time: (22500,) min: 0.5 max: 447.2063
  event: (22500,) {np.int64(0): np.int64(11250), np.int64(1): np.int64(5613), np.int64(2): np.int64(5637)}

test:
  X: (22500, 12)
  time: (2250

In [5]:
import torch
from src.losses import CoxPHLoss

loss_fn = CoxPHLoss()

log_risk = torch.tensor([0.2, 1.0, -0.5, 0.7])
time = torch.tensor([5.0, 2.0, 8.0, 4.0])
event = torch.tensor([1, 1, 0, 1])

loss = loss_fn(log_risk, time, event)
print(loss)

tensor(0.6434)


In [6]:
import torch
from src.models.baselines import DeepSurv

model = DeepSurv(input_dim=12)

x = torch.randn(32, 12)
log_risk = model(x)

print(log_risk.shape)
print(log_risk[:5])

torch.Size([32])
tensor([ 2.8043e-01, -6.6613e-01, -1.5796e-03,  4.4346e-01,  2.3218e+00],
       grad_fn=<SliceBackward0>)


In [7]:
import numpy as np
from src.evaluation import c_index

time = np.array([5, 2, 8, 4])
event = np.array([1, 1, 0, 1])
risk = np.array([0.2, 1.0, -0.5, 0.7])

print(c_index(time, event, risk))

1.0


In [8]:
import torch

from src.datasets.loaders import load_dataset
from src.models.baselines import DeepSurv
from src.training.baseline_runner import train_deepsurv

ds = load_dataset("synthetic_binary", data_root="../data")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

model = DeepSurv(
    input_dim=ds["n_features"],
    hidden_dims=[64, 32],
    dropout=0.2,
)

model, history = train_deepsurv(
    model=model,
    train_data=ds["train"],
    val_data=ds["val"],
    device=device,
    batch_size=512,
    lr=1e-3,
    epochs=5,
)

cpu
Epoch 001 | train_loss=4.7863 | val_cindex=0.7738
Epoch 002 | train_loss=4.6582 | val_cindex=0.7777
Epoch 003 | train_loss=4.6329 | val_cindex=0.7795
Epoch 004 | train_loss=4.6223 | val_cindex=0.7797
Epoch 005 | train_loss=4.6139 | val_cindex=0.7802


In [9]:
import torch
from src.models.baselines import DeepHit

model = DeepHit(
    input_dim=12,
    n_time_bins=50,
    n_events=1,
)

x = torch.randn(32, 12)
logits = model(x)

print(logits.shape)

torch.Size([32, 1, 50])


In [10]:
import torch
from src.losses import DeepHitLoss

loss_fn = DeepHitLoss(alpha=1.0, beta=0.2, sigma=0.1)

logits = torch.randn(32, 1, 50)
time_bin = torch.randint(0, 50, (32,))
event = torch.randint(0, 2, (32,))

loss = loss_fn(logits, time_bin, event)
print(loss)

tensor(2.5121)


In [11]:
from src.datasets.loaders import load_dataset, add_time_bins

ds = load_dataset("synthetic_binary", data_root="../data")
ds = add_time_bins(ds, n_time_bins=50, method="quantile")

print(ds["n_time_bins"])
print(ds["train"]["time_bin"].min(), ds["train"]["time_bin"].max())
print(ds["val"]["time_bin"].min(), ds["val"]["time_bin"].max())
print(ds["test"]["time_bin"].min(), ds["test"]["time_bin"].max())

39
0 38
0 38
0 38


In [12]:
from src.datasets.loaders import load_dataset
from src.graphs.builders import build_knn_graph

ds = load_dataset("synthetic_binary", data_root="../data")

X_all = np.concatenate([
    ds["train"]["X"],
    ds["val"]["X"],
    ds["test"]["X"],
], axis=0)

edge_index = build_knn_graph(X_all, k=10)

print("X_all:", X_all.shape)
print("edge_index:", edge_index.shape)
print("min node:", edge_index.min())
print("max node:", edge_index.max())

X_all: (150000, 12)
edge_index: (2, 2254902)
min node: 0
max node: 149999


In [13]:
from src.datasets.loaders import load_dataset
from src.datasets.graph_dataset import make_graph_survival_data

ds = load_dataset("synthetic_binary", data_root="../data")

data = make_graph_survival_data(
    dataset=ds,
    edge_index_path="../data/synthetic_binary/graphs/knn_k10_euclidean_edge_index.npy",
)

print(data)
print(data.x.shape)
print(data.edge_index.shape)
print(data.train_mask.sum(), data.val_mask.sum(), data.test_mask.sum())

Data(x=[150000, 12], edge_index=[2, 2254902], time=[150000], event=[150000], train_mask=[150000], val_mask=[150000], test_mask=[150000])
torch.Size([150000, 12])
torch.Size([2, 2254902])
tensor(105000) tensor(22500) tensor(22500)


In [14]:
from src.models.encoders import GraphSAGEEncoder

encoder = GraphSAGEEncoder(input_dim=data.x.shape[1], hidden_dims=[64, 32])
out = encoder(data.x, data.edge_index)

print(out.shape)

torch.Size([150000, 32])


In [15]:
from src.models.gnn_models import GraphSAGECoxModel

model = GraphSAGECoxModel(
    input_dim=data.x.shape[1],
    hidden_dims=[64, 32],
    dropout=0.2,
)

log_risk = model(data.x, data.edge_index)

print(log_risk.shape)

torch.Size([150000])


In [10]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

import importlib
import src.ssl.pseudo_labeling as pl
importlib.reload(pl)

print(dir(pl))

['F', 'Optional', 'PseudoLabelResult', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'dataclass', 'deephit_logits_to_probs', 'deephit_pseudo_label_loss', 'generate_deephit_pseudo_labels', 'generate_pseudo_labels_from_model', 'make_censored_candidate_mask', 'torch']


In [17]:
from src.models.gnn_models import GraphSAGEDeepHitModel

model = GraphSAGEDeepHitModel(
    input_dim=data.x.shape[1],
    n_time_bins=39,
    n_events=1,
    hidden_dims=[64, 32],
)

logits = model(data.x, data.edge_index)

print(logits.shape)

torch.Size([150000, 1, 39])


In [18]:
from src.datasets.loaders import load_dataset, add_time_bins
from src.datasets.graph_dataset import make_graph_survival_data

ds = load_dataset("synthetic_binary", data_root="../data")
ds = add_time_bins(ds, n_time_bins=50, method="quantile")

data = make_graph_survival_data(
    dataset=ds,
    edge_index_path="../data/synthetic_binary/graphs/knn_k10_euclidean_edge_index.npy",
)

print(data)
print(data.time_bin.shape)
print(data.time_bin.min(), data.time_bin.max())

Data(x=[150000, 12], edge_index=[2, 2254902], time=[150000], event=[150000], train_mask=[150000], val_mask=[150000], test_mask=[150000], time_bin=[150000])
torch.Size([150000])
tensor(0) tensor(38)


In [5]:
import torch

from src.ssl.pseudo_labeling import (
    generate_deephit_pseudo_labels,
    make_censored_candidate_mask,
)

# fake DeepHit teacher output
N = 8
K = 2
T = 5

logits = torch.randn(N, K, T)

event = torch.tensor([0, 1, 0, 2, 0, 0, 1, 0])
train_mask = torch.tensor([True, True, True, True, False, True, True, True])

candidate_mask = make_censored_candidate_mask(event, train_mask)

pseudo = generate_deephit_pseudo_labels(
    logits=logits,
    candidate_mask=candidate_mask,
    min_confidence=0.2,
)

print("pseudo_probs:", pseudo.pseudo_probs.shape)
print("confidence:", pseudo.confidence.shape)
print("selected_mask:", pseudo.selected_mask)
print("pred_event:", pseudo.pred_event)
print("pred_time_bin:", pseudo.pred_time_bin)

print("probability sums:", pseudo.pseudo_probs.sum(dim=(1, 2)))
print("selected count:", pseudo.selected_mask.sum().item())

pseudo_probs: torch.Size([8, 2, 5])
confidence: torch.Size([8])
selected_mask: tensor([ True, False,  True, False, False,  True, False,  True])
pred_event: tensor([1, 1, 1, 1, 1, 1, 0, 1])
pred_time_bin: tensor([1, 4, 0, 3, 4, 2, 0, 2])
probability sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])
selected count: 4


In [8]:
import torch

from src.ssl.pseudo_labeling import (
    generate_deephit_pseudo_labels,
    make_censored_candidate_mask,
    deephit_pseudo_label_loss,
)

N = 8
K = 2
T = 5

teacher_logits = torch.randn(N, K, T)
student_logits = torch.randn(N, K, T, requires_grad=True)

event = torch.tensor([0, 1, 0, 2, 0, 0, 1, 0])
train_mask = torch.tensor([True, True, True, True, False, True, True, True])

candidate_mask = make_censored_candidate_mask(event, train_mask)

pseudo = generate_deephit_pseudo_labels(
    logits=teacher_logits,
    candidate_mask=candidate_mask,
    min_confidence=0.2,
)

loss = deephit_pseudo_label_loss(
    student_logits=student_logits,
    pseudo_probs=pseudo.pseudo_probs,
    selected_mask=pseudo.selected_mask,
)

print("pseudo loss:", loss.item())
loss.backward()
print("grad exists:", student_logits.grad is not None)

pseudo loss: 0.8384825587272644
grad exists: True


In [11]:
from src.ssl.pseudo_labeling import generate_pseudo_labels_from_model